# Viager Europe scraper and valuation notebook

This notebook:
- scrapes all listing pages from the English Viager Europe catalogue
- follows each advert detail page to enrich the dataset
- detects resident ages and the `homme` / `femme` icons as male / female
- estimates remaining life expectancy for each resident
- builds a simple present-value valuation model for the implied discount
- saves the enriched dataset into `viager_data/viager_europe_listings_enriched_<date>.json`

**Important note:** the valuation section is a heuristic market model for exploration, not legal, actuarial, or investment advice.

In [1]:
from __future__ import annotations

import json
import math
import re
import time
from datetime import datetime, UTC
from pathlib import Path
from urllib.parse import urljoin

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display

BASE_URL = "https://www.viager-europe.com/en/annonces/"
ROOT_DIR = Path.cwd()
OUTPUT_DIR = ROOT_DIR / "viager_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DATE = datetime.now(UTC).strftime("%Y%m%d")
OUTPUT_JSON = OUTPUT_DIR / f"viager_europe_listings_enriched_{RUN_DATE}.json"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
}

session = requests.Session()
session.headers.update(HEADERS)

ANNUAL_DISCOUNT_RATE = 0.04

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)

_lint_keepalive = (json, math, re, time, urljoin, plt, np, BeautifulSoup, Markdown, display)
assert len(_lint_keepalive) == 10

print(f"Output JSON will be written to: {OUTPUT_JSON}")

Output JSON will be written to: /home/endrem/project/calculate_finn/cars/random_stuff/viager_data/viager_europe_listings_enriched_20260324.json


In [ ]:
def clean_text(value: str | None) -> str:
    if not value:
        return ""
    return re.sub(r"\s+", " ", value).strip()


def parse_money(value: str | None) -> float | None:
    if not value:
        return None
    match = re.search(r"([\d\s]+(?:,\d+)?)\s*€", value.replace("\xa0", " "))
    if not match:
        return None
    normalized = match.group(1).replace(" ", "").replace(",", ".")
    try:
        return float(normalized)
    except ValueError:
        return None


def parse_percent(value: str | None) -> float | None:
    if not value:
        return None
    match = re.search(r"(\d+(?:[.,]\d+)?)\s*%", value)
    if not match:
        return None
    return float(match.group(1).replace(",", "."))


def extract_money_after_label(text: str, label: str) -> float | None:
    pattern = rf"{re.escape(label)}\s+([\d\s]+(?:,\d+)?)\s*€"
    match = re.search(pattern, text, flags=re.IGNORECASE)
    if not match:
        return None
    return parse_money(match.group(0))


def extract_text_after_label(text: str, label: str, stop_tokens: list[str]) -> str | None:
    start = re.search(re.escape(label), text, flags=re.IGNORECASE)
    if not start:
        return None
    remainder = text[start.end():].strip()
    stop_positions = [re.search(re.escape(token), remainder, flags=re.IGNORECASE) for token in stop_tokens]
    valid_stops = [m.start() for m in stop_positions if m]
    snippet = remainder[: min(valid_stops)] if valid_stops else remainder
    return clean_text(snippet) or None


def parse_contract_years(text: str) -> int | None:
    match = re.search(r"FOR SALE IN\s+(\d+)\s+YEARS", text, flags=re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None


def parse_gender_from_img(src: str | None) -> str | None:
    source = (src or "").lower()
    if "homme" in source:
        return "male"
    if "femme" in source:
        return "female"
    return None


LIFE_EXPECTANCY_TABLE = {
    "male": {60: 23.0, 65: 19.3, 70: 15.7, 75: 12.4, 80: 9.5, 85: 6.9, 90: 4.8, 95: 3.2, 100: 2.2},
    "female": {60: 27.1, 65: 23.0, 70: 19.0, 75: 15.1, 80: 11.6, 85: 8.5, 90: 5.9, 95: 3.9, 100: 2.5},
}


def remaining_life_expectancy(age: float | int | None, sex: str | None) -> float | None:
    if age is None or sex not in LIFE_EXPECTANCY_TABLE:
        return None
    anchors = np.array(sorted(LIFE_EXPECTANCY_TABLE[sex].keys()), dtype=float)
    values = np.array([LIFE_EXPECTANCY_TABLE[sex][age_key] for age_key in anchors], dtype=float)
    age_value = float(age)
    if age_value <= anchors[0]:
        slope = (values[1] - values[0]) / (anchors[1] - anchors[0])
        return float(values[0] + (age_value - anchors[0]) * slope)
    if age_value >= anchors[-1]:
        return float(values[-1])
    return float(np.interp(age_value, anchors, values))


def monthly_annuity_present_value(monthly_payment: float | None, years: float | None, annual_rate: float = ANNUAL_DISCOUNT_RATE) -> float:
    if not monthly_payment or monthly_payment <= 0:
        return 0.0
    if not years or years <= 0 or (math.isnan(years) if isinstance(years, float) else False):
        return 0.0
    monthly_rate = (1 + annual_rate) ** (1 / 12) - 1
    periods = max(int(round(years * 12)), 1)
    return float(monthly_payment * (1 - (1 + monthly_rate) ** (-periods)) / monthly_rate)


def theoretical_total_price(property_value: float | None, years: float | None, annual_rate: float = ANNUAL_DISCOUNT_RATE) -> float | None:
    if not property_value or property_value <= 0:
        return None
    horizon = max(float(years or 0), 0.0)
    return float(property_value / ((1 + annual_rate) ** horizon))


def resident_summary(residents: list[dict]) -> str:
    if not residents:
        return "No resident data"
    parts = []
    for resident in residents:
        sex = resident.get("sex", "unknown")
        age = resident.get("age")
        expectancy = resident.get("remaining_life_expectancy_years")
        if age is None:
            continue
        if expectancy is None:
            parts.append(f"{sex}: {age}y")
        else:
            parts.append(f"{sex}: {age}y (≈ {expectancy:.1f}y left)")
    return ", ".join(parts) if parts else "No resident data"


def estimate_occupation_horizon(residents: list[dict], viager_type: str | None, contract_years: int | None) -> float:
    if contract_years:
        return float(contract_years)
    if viager_type and "libre" in viager_type.lower():
        return 0.0
    remaining_years = [resident.get("remaining_life_expectancy_years") for resident in residents if resident.get("remaining_life_expectancy_years") is not None]
    if not remaining_years:
        return float("nan")
    return float(max(remaining_years))


def safe_float(value) -> float | None:
    if value is None:
        return None
    if isinstance(value, float) and math.isnan(value):
        return None
    return float(value) if isinstance(value, (int, float, np.integer, np.floating)) else value

In [ ]:
def get_soup(url: str) -> BeautifulSoup:
    response = session.get(url, timeout=30)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


def infer_total_pages(first_page_html: str) -> int:
    total_pages_match = re.search(r'"total_pages":(\d+)', first_page_html)
    if total_pages_match:
        return int(total_pages_match.group(1))
    page_numbers = [int(value) for value in re.findall(r"/page/(\d+)/", first_page_html)]
    return max(page_numbers) if page_numbers else 1


def extract_listing_cards(soup: BeautifulSoup) -> list[BeautifulSoup]:
    cards: list[BeautifulSoup] = []
    seen_urls: set[str] = set()
    for anchor in soup.find_all("a", href=lambda href: href and "/en/annonces-details/" in href):
        detail_url = urljoin(BASE_URL, anchor.get("href"))
        if detail_url in seen_urls:
            continue
        container = anchor
        while container and getattr(container, "name", None) != "body":
            text = clean_text(container.get_text(" ", strip=True))
            if "See advert" in text and ("Lump-sum payment" in text or "Initial down payment" in text or "Annuity" in text):
                cards.append(container)
                seen_urls.add(detail_url)
                break
            container = container.parent
    return cards


def parse_listing_card(card: BeautifulSoup) -> dict:
    text = clean_text(card.get_text(" ", strip=True))
    detail_anchor = card.find("a", href=lambda href: href and "/en/annonces-details/" in href)
    detail_url = urljoin(BASE_URL, detail_anchor.get("href")) if detail_anchor else None

    reference_match = re.search(r"\b([A-Z]{1,3}-\d{2,4}-EN)\b", text)
    reference = reference_match.group(1) if reference_match else None
    before_reference = clean_text(text.split(reference, 1)[0]) if reference else ""
    after_reference = clean_text(text.split(reference, 1)[1]) if reference else text

    cutoff_tokens = ["Buying price", "Property value", "See advert"]
    first_cut_positions = [after_reference.find(token) for token in cutoff_tokens if token in after_reference]
    header_block = after_reference[: min(first_cut_positions)] if first_cut_positions else after_reference
    header_match = re.search(r"^(.*?)\s*-\s*(\d{5})\s+(.*)$", header_block)
    city = clean_text(header_match.group(1)) if header_match else None
    postal_code = header_match.group(2) if header_match else None
    property_block = clean_text(header_match.group(3)) if header_match else header_block

    rooms_match = re.search(r"(\d+)\s+rooms?", property_block, flags=re.IGNORECASE)
    rooms = int(rooms_match.group(1)) if rooms_match else None
    property_type = clean_text(re.sub(r"\b\d+\s+rooms?\b", "", property_block, flags=re.IGNORECASE)) or None

    result = {
        "listing_type": before_reference.title() if before_reference else None,
        "reference": reference,
        "city": city,
        "postal_code": postal_code,
        "property_type": property_type,
        "rooms": rooms,
        "buying_price_eur": extract_money_after_label(text, "Buying price"),
        "discount_pct_listed": parse_percent(extract_text_after_label(text, "Discount", ["Property value", "See advert"])),
        "property_value_eur": extract_money_after_label(text, "Property value"),
        "lump_sum_payment_eur": extract_money_after_label(text, "Lump-sum payment"),
        "initial_down_payment_eur": extract_money_after_label(text, "Initial down payment"),
        "annuity_monthly_eur": extract_money_after_label(text, "Annuity"),
        "contract_term_years": parse_contract_years(text),
        "status_text": next((label for label in ["EXCLUSIVITY", "UNDER COMPROMISE", "TO SEIZE !"] if label in text.upper()), None),
        "detail_url": detail_url,
        "card_text": text,
    }

    listed_ages = [int(age) for age in re.findall(r"(\d{2,3})\s+years", text, flags=re.IGNORECASE)]
    result["listed_ages"] = listed_ages
    return result


def extract_description(soup: BeautifulSoup) -> str | None:
    heading = soup.find(lambda tag: tag.name in {"h2", "h3", "h4"} and "description of the property" in clean_text(tag.get_text(" ", strip=True)).lower())
    if not heading:
        return None
    parts: list[str] = []
    for node in heading.find_all_next():
        if node == heading:
            continue
        if getattr(node, "name", None) in {"h1", "h2", "h3"}:
            title = clean_text(node.get_text(" ", strip=True)).lower()
            if any(stop in title for stop in ["your contact", "location"]):
                break
        if getattr(node, "name", None) in {"p", "div"}:
            text = clean_text(node.get_text(" ", strip=True))
            if len(text) >= 40 and text not in parts:
                parts.append(text)
        if len(" ".join(parts)) >= 2500:
            break
    return clean_text(" ".join(parts)) or None


def parse_detail_page(detail_url: str) -> dict:
    soup = get_soup(detail_url)
    page_title = clean_text(soup.title.get_text(" ", strip=True)) if soup.title else None
    heading = soup.find(["h1", "h2"], string=True)
    advert_heading = clean_text(heading.get_text(" ", strip=True)) if heading else None

    residents: list[dict] = []
    for image in soup.find_all("img", src=lambda src: src and "Shape-" in src):
        sex = parse_gender_from_img(image.get("src"))
        if sex is None:
            continue
        parent_text = clean_text(image.parent.get_text(" ", strip=True)) if image.parent else ""
        if not parent_text:
            sibling = image.find_next(string=re.compile(r"\d{2,3}\s*years", flags=re.IGNORECASE))
            parent_text = clean_text(str(sibling)) if sibling else ""
        age_match = re.search(r"(\d{2,3})\s*years", parent_text, flags=re.IGNORECASE)
        age = int(age_match.group(1)) if age_match else None
        residents.append({
            "sex": sex,
            "age": age,
        })

    unique_residents: list[dict] = []
    seen_residents: set[tuple] = set()
    for resident in residents:
        marker = (resident.get("sex"), resident.get("age"))
        if marker in seen_residents:
            continue
        seen_residents.add(marker)
        resident["remaining_life_expectancy_years"] = remaining_life_expectancy(resident.get("age"), resident.get("sex"))
        resident["expected_age_at_death"] = (
            resident["age"] + resident["remaining_life_expectancy_years"]
            if resident.get("age") is not None and resident.get("remaining_life_expectancy_years") is not None
            else None
        )
        unique_residents.append(resident)

    gallery_images = []
    for image in soup.find_all("img", src=lambda src: src and "/wp-content/uploads/" in src):
        src = image.get("src")
        if src and src not in gallery_images:
            gallery_images.append(src)

    detail_text = clean_text(soup.get_text(" ", strip=True))
    property_size_match = re.search(r"(\d+(?:[.,]\d+)?)\s*m²", detail_text)
    return {
        "page_title": page_title,
        "advert_heading": advert_heading,
        "property_size_sqm": float(property_size_match.group(1).replace(",", ".")) if property_size_match else None,
        "description": extract_description(soup),
        "gallery_image_urls": gallery_images[:20],
        "residents": unique_residents,
        "detail_page_text": detail_text[:5000],
    }


def scrape_all_listings(max_pages: int | None = None, pause_seconds: float = 0.35) -> list[dict]:
    first_response = session.get(BASE_URL, timeout=30)
    first_response.raise_for_status()
    total_pages = infer_total_pages(first_response.text)
    if max_pages is not None:
        total_pages = min(total_pages, max_pages)

    print(f"Estimated {total_pages} pages to scrape")

    records: list[dict] = []
    seen_references: set[str] = set()

    for page_number in range(1, total_pages + 1):
        page_url = BASE_URL if page_number == 1 else urljoin(BASE_URL, f"page/{page_number}/")
        try:
            soup = get_soup(page_url)
            cards = extract_listing_cards(soup)
            print(f"Page {page_number}/{total_pages}: found {len(cards)} listing cards")

            for card in cards:
                listing = parse_listing_card(card)
                reference = listing.get("reference")
                if reference and reference in seen_references:
                    continue
                if reference:
                    seen_references.add(reference)

                detail_url = listing.get("detail_url")
                detail_data = parse_detail_page(detail_url) if detail_url else {}
                record = {**listing, **detail_data}
                records.append(record)
                time.sleep(pause_seconds)
        except Exception as e:
            print(f"  ⚠ Error on page {page_number}: {type(e).__name__}: {str(e)[:100]}")
            continue

        time.sleep(pause_seconds)

    return records

In [ ]:
print("Starting scrape of all listing pages...")
records = scrape_all_listings()
print(f"\nScraped {len(records)} total listing records\n")

if not records:
    display(Markdown("### ⚠️ No listings were found"))
    display(Markdown("The scraper found 0 records. This could mean:"))
    display(Markdown("- The website structure has changed"))
    display(Markdown("- Network timeout or connection issue"))
    display(Markdown("- The page requires JavaScript to load (requires Selenium)"))
    display(Markdown("\n**Next steps:** Check the error messages above or try running again."))
    raise RuntimeError("Scraper returned empty results")

df = pd.DataFrame(records)

df["cash_component_eur"] = df["lump_sum_payment_eur"].fillna(df["initial_down_payment_eur"])
df["resident_count"] = df["residents"].apply(lambda values: len(values) if isinstance(values, list) else 0)
df["male_count"] = df["residents"].apply(lambda values: sum(1 for resident in values if resident.get("sex") == "male") if isinstance(values, list) else 0)
df["female_count"] = df["residents"].apply(lambda values: sum(1 for resident in values if resident.get("sex") == "female") if isinstance(values, list) else 0)
df["resident_summary"] = df["residents"].apply(lambda values: resident_summary(values) if isinstance(values, list) else "No resident data")
df["expected_occupation_years"] = df.apply(
    lambda row: estimate_occupation_horizon(row.get("residents") or [], row.get("listing_type"), row.get("contract_term_years")),
    axis=1,
 )

df["annuity_pv_eur"] = df.apply(
    lambda row: monthly_annuity_present_value(
        row.get("annuity_monthly_eur"),
        row.get("expected_occupation_years"),
        ANNUAL_DISCOUNT_RATE,
    ),
    axis=1,
 )

df["theoretical_total_price_eur"] = df.apply(
    lambda row: theoretical_total_price(
        row.get("property_value_eur"),
        row.get("expected_occupation_years"),
        ANNUAL_DISCOUNT_RATE,
    ),
    axis=1,
 )

df["theoretical_lump_sum_eur"] = (df["theoretical_total_price_eur"] - df["annuity_pv_eur"]).clip(lower=0)
df["theoretical_discount_pct"] = (1 - (df["theoretical_total_price_eur"] / df["property_value_eur"])) * 100
df["actual_total_price_eur"] = df["cash_component_eur"].fillna(0) + df["annuity_pv_eur"].fillna(0)
df.loc[df["cash_component_eur"].isna() & df["annuity_monthly_eur"].isna(), "actual_total_price_eur"] = np.nan
df["actual_discount_pct_modelled"] = (1 - (df["actual_total_price_eur"] / df["property_value_eur"])) * 100
df["pricing_gap_vs_model_eur"] = df["actual_total_price_eur"] - df["theoretical_total_price_eur"]
df["pricing_gap_pct_of_property_value"] = (df["pricing_gap_vs_model_eur"] / df["property_value_eur"]) * 100

numeric_columns = [
    "buying_price_eur",
    "discount_pct_listed",
    "property_value_eur",
    "lump_sum_payment_eur",
    "initial_down_payment_eur",
    "annuity_monthly_eur",
    "contract_term_years",
    "property_size_sqm",
    "cash_component_eur",
    "resident_count",
    "male_count",
    "female_count",
    "expected_occupation_years",
    "annuity_pv_eur",
    "theoretical_total_price_eur",
    "theoretical_lump_sum_eur",
    "theoretical_discount_pct",
    "actual_total_price_eur",
    "actual_discount_pct_modelled",
    "pricing_gap_vs_model_eur",
    "pricing_gap_pct_of_property_value",
    "rooms",
 ]

for column in numeric_columns:
    if column in df.columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

def dataframe_to_serializable_records(frame: pd.DataFrame) -> list[dict]:
    serializable_records = []
    for record in frame.to_dict(orient="records"):
        cleaned = {}
        for key, value in record.items():
            if isinstance(value, list):
                cleaned[key] = value
            elif isinstance(value, dict):
                cleaned[key] = value
            elif isinstance(value, (np.floating, float)) and pd.isna(value):
                cleaned[key] = None
            elif isinstance(value, (np.integer, int)):
                cleaned[key] = int(value)
            elif isinstance(value, (np.floating, float)):
                cleaned[key] = float(value)
            else:
                cleaned[key] = value
        serializable_records.append(cleaned)
    return serializable_records

serialized_records = dataframe_to_serializable_records(df)
with OUTPUT_JSON.open("w", encoding="utf-8") as file_handle:
    json.dump(serialized_records, file_handle, ensure_ascii=False, indent=2)

summary = {
    "listing_count": int(len(df)),
    "pages_scraped_estimate": int(df["detail_url"].nunique() // 20 + (1 if len(df) % 20 else 0)),
    "with_resident_data": int((df["resident_count"] > 0).sum()),
    "with_annuity": int(df["annuity_monthly_eur"].notna().sum()),
    "output_json": str(OUTPUT_JSON),
}

display(Markdown("## Scrape completed"))
display(pd.DataFrame([summary]))
df.head(3)

Page 1/15: found 36 listing cards
Page 2/15: found 32 listing cards
Page 3/15: found 33 listing cards
Page 4/15: found 29 listing cards
Page 5/15: found 26 listing cards
Page 6/15: found 25 listing cards
Page 7/15: found 22 listing cards
Page 8/15: found 28 listing cards
Page 9/15: found 0 listing cards
Page 10/15: found 21 listing cards
Page 11/15: found 0 listing cards
Page 12/15: found 0 listing cards
Page 13/15: found 0 listing cards
Page 14/15: found 0 listing cards
Page 15/15: found 0 listing cards


ValueError: cannot convert float NaN to integer

## Reading the valuation columns

The model uses the following assumptions:
- resident life expectancy is estimated from the detected age and `homme`/`femme` icon
- for multiple residents, the occupation horizon uses the **longest** expected remaining life
- if the card explicitly says `FOR SALE IN X YEARS`, that term overrides life expectancy
- the model discounts future vacant possession at **4% per year**
- if there is a monthly annuity, its present value is added to the effective price paid by the buyer

Key columns:
- `expected_occupation_years` = estimated years until vacant possession or contract maturity
- `theoretical_total_price_eur` = fair present value of the occupied property
- `theoretical_discount_pct` = discount implied by that present value
- `actual_total_price_eur` = cash component plus present value of the monthly annuity
- `pricing_gap_vs_model_eur` = positive means the advert is priced above the model, negative means below the model

In [ ]:
presentation_columns = [
    "reference",
    "listing_type",
    "city",
    "postal_code",
    "property_type",
    "rooms",
    "property_value_eur",
    "cash_component_eur",
    "annuity_monthly_eur",
    "resident_summary",
    "expected_occupation_years",
    "discount_pct_listed",
    "theoretical_discount_pct",
    "actual_discount_pct_modelled",
    "pricing_gap_vs_model_eur",
    "detail_url",
 ]

best_value = (
    df[presentation_columns]
    .sort_values(["pricing_gap_vs_model_eur", "property_value_eur"], ascending=[True, False])
    .head(20)
    .copy()
 )

display(Markdown("## 20 listings that look cheapest versus the model"))
display(
    best_value.style.format(
        {
            "property_value_eur": "{:,.0f} €",
            "cash_component_eur": "{:,.0f} €",
            "annuity_monthly_eur": "{:,.0f} € / month",
            "expected_occupation_years": "{:.1f} years",
            "discount_pct_listed": "{:.1f}%",
            "theoretical_discount_pct": "{:.1f}%",
            "actual_discount_pct_modelled": "{:.1f}%",
            "pricing_gap_vs_model_eur": "{:,.0f} €",
        },
        na_rep="—",
    )
    .background_gradient(subset=["pricing_gap_vs_model_eur"], cmap="RdYlGn_r")
 )

resident_view = df[[
    "reference",
    "city",
    "listing_type",
    "resident_summary",
    "male_count",
    "female_count",
    "expected_occupation_years",
    "property_value_eur",
    "theoretical_discount_pct",
    "cash_component_eur",
    "annuity_monthly_eur",
 ]].copy()

display(Markdown("## Resident life expectancy view"))
display(
    resident_view.sort_values(["expected_occupation_years", "property_value_eur"], ascending=[False, False]).head(20).style.format(
        {
            "expected_occupation_years": "{:.1f} years",
            "property_value_eur": "{:,.0f} €",
            "theoretical_discount_pct": "{:.1f}%",
            "cash_component_eur": "{:,.0f} €",
            "annuity_monthly_eur": "{:,.0f} € / month",
        },
        na_rep="—",
    )
 )

In [ ]:
plot_df = df.dropna(subset=["property_value_eur", "theoretical_discount_pct"]).copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(
    plot_df["expected_occupation_years"],
    plot_df["theoretical_discount_pct"],
    s=np.clip(plot_df["property_value_eur"] / 4000, 20, 350),
    alpha=0.65,
    c=plot_df["pricing_gap_vs_model_eur"].fillna(0),
    cmap="coolwarm",
 )
axes[0].set_title("Expected occupation years vs theoretical discount")
axes[0].set_xlabel("Expected occupation years")
axes[0].set_ylabel("Theoretical discount (%)")
axes[0].grid(alpha=0.25)

top_gap = df.dropna(subset=["pricing_gap_vs_model_eur"]).copy()
top_gap = top_gap.sort_values("pricing_gap_vs_model_eur").head(12)

axes[1].barh(
    top_gap["reference"].astype(str),
    top_gap["pricing_gap_vs_model_eur"],
    color=["#2ca02c" if value < 0 else "#d62728" for value in top_gap["pricing_gap_vs_model_eur"]],
 )
axes[1].set_title("Most attractive listings vs model")
axes[1].set_xlabel("Pricing gap vs model (€)")
axes[1].set_ylabel("Reference")
axes[1].axvline(0, color="black", linewidth=1)
axes[1].grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()

print(f"Saved {len(df):,} enriched listings to {OUTPUT_JSON}")